# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

import skewerjax.optical_depth.optical_depth as od

In [ ]:
from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian

# Validation against analytical result with $T=0,\, v_{LoS}$=0

In [ ]:
L_box = 1.0

T0 = jnp.finfo(jnp.dtype('float32')).tiny
v0 = 0.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))
    }
})

In [ ]:
scale_factor = 1.0

points = jnp.array([scale_factor, L_box/2, L_box/2, L_box/2])[None, :]
LoS_directions = jnp.array([[0.0, 0.0, 1.0]])

scale_factor_obs = 1.0

wavelengths = jnp.linspace(970, 1230, 2**20)  # in Angstroms

line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

In [ ]:
phi_ns = od.compute_line_profiles(
    points,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    compute_mode="pmap",
    voigt_implementation_mode = "ExoJAX",
    num_gpus = None
)[:, 0]

### Analytic benchmark

In [ ]:
gamma_GHz = line_bundle.A21_GHz / (4.0 * jnp.pi)

lambdas_A_prime = od.compute_lambda_prime(
    wavelengths,
    scale_factor_obs,
    scale_factor=jnp.array([points[0, 0]]),
    LoS_velocity=jnp.zeros(line_bundle.n_lines)[:, None]
)[:, 0]

lambda0_A = jnp.asarray(line_bundle.lambda0_A)  # (N_lines,)
nu_GHz = ((1.0 / lambdas_A_prime) - (1.0 / lambda0_A[:, None])) * phys.c * 10.0

phi_analytic = Lorentzian(nu_GHz, gamma_GHz[:, None])

### Validation plot

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                    # (N_lines, N_lambda)
phi_analytic = np.asarray(phi_analytic)        # (N_lines, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:line_bundle.n_lines])

fig, ax = plt.subplots(figsize=(14, 6))

for i_sp, line in enumerate(line_bundle.lines):
    ax.scatter(
        np.array([line_bundle.lines[i_sp].lambda0_A]), 
        np.array([4./line_bundle.lines[i_sp].A21_GHz]), 
        marker="x", color=colors[i_sp], s=100, alpha=1.0
    )
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ax.plot(lambdas_np, phi_np[i_sp, :], color=colors[i_sp], linestyle='-', linewidth=2.0)
    ax.plot(lambdas_np, phi_analytic[i_sp, :], color=colors[i_sp], linestyle=':', linewidth=4.0, alpha=0.6)

ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

handles = [
    Line2D([0], [0], color='grey', lw=4, linestyle='-', label='SkewerJAX'),
    Line2D([0], [0], color='grey', lw=4, linestyle=':', label='Analytical'),
    Line2D([0], [0], color='grey', marker='x', linestyle='None', markersize=10, 
            label=r'Peak value $\frac{4}{A_{21}}$')
]
legend = ax.legend(handles=handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(legend)

handles = []
for i_sp, line in enumerate(line_bundle.lines):
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    handles.append(Line2D([0], [0], color=colors[i_sp], lw=3, linestyle='-', label=lbl))
leg_L = ax.legend(handles=handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()





# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
N_lines = line_bundle.n_lines

fig, axes = plt.subplots(
    nrows=2, ncols=N_lines, 
    figsize=(14, 8), 
    sharex='col',
    sharey='row',
    gridspec_kw=dict(height_ratios=[2, 1])
)

if N_lines == 1:
    axes = axes.reshape(2, 1)

for i_sp, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):
    lambda0 = line.lambda0_A
    
    ax_main.scatter(
        np.array([lambda0]), 
        np.array([4./line_bundle.lines[i_sp].A21_GHz]), 
        marker="x", color=colors[i_sp], s=100, alpha=1.0
    )
    ax_main.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)
    
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]
    
    ax_main.plot(lam_win, phi_np[i_sp, mask], color=colors[i_sp], linestyle='-', linewidth=2.0, label='SkewerJAX')
    ax_main.plot(lam_win, phi_analytic[i_sp, mask], color=colors[i_sp], linestyle=':', linewidth=4.0, alpha=0.6, label='Analytical')
    
    ax_main.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_main.set_yscale("log")
    ax_main.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
    
    residual = phi_np[i_sp, mask] / phi_analytic[i_sp, mask] - 1
    
    ax_res.plot(lam_win, residual, color=colors[i_sp], linestyle='-', linewidth=2.0)
    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
    
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax_res.set_ylabel("residual")
    ax_res.set_yscale("symlog", linthresh=1e-6)
    
    major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
    ax_res.set_yticks(major_ticks)
    ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
    ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)

plt.show()

# Check variations with temperature and LoS velocity

In [ ]:
L_box = 1.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-0.001, v0_max=0.001, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))
    }
})

In [ ]:
scale_factor = 1.0

N_sk = 7
N_ppsk = 5
N_grid = 64

points = aux.make_plane_points(L_box, 'y', L_box/2, scale_factor, N_sk, N_ppsk)
LoS_directions = aux.make_LoS(points.shape[:-1], [1.0, 0.0, 0.0])

scale_factor_obs = 1.0

wavelengths = jnp.linspace(970, 1280, 2**18)  # in Angstroms

line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

In [ ]:
phi_ns = od.compute_line_profiles(
    points,
    LoS_directions,
    scale_factor_obs,
    wavelengths,
    line_bundle,
    fields,
    compute_mode="pmap",
    voigt_implementation_mode = "ExoJAX",
    num_gpus = None
)

### Plot showing temperature variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
phi_np = np.asarray(phi_ns)                     # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_ppsk = N_ppsk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for temperatures (colors)
temp_handles = []
for i_sk in range(N_sk):
    _sampled_T = fields.temperature(points[i_sk])
    lbl = rf"$T = {_sampled_T[0, 0]:.2g}\,\mathrm{{K}}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()



# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_sk in range(N_sk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()

### Plot showing $v_{LoS}$ variations

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                          # (N_lambda,)
phi_np = np.asarray(phi_ns)                                    # (N_sp, N_sk, N_ppsk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

i_sk = N_sk // 2

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, :]   # full wavelength range
        ax.plot(lambdas_np, profile, color=colors[i_ppsk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_yscale("log")
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# Legend for velocities (colors)
temp_handles = []
for i_ppsk in range(N_ppsk):
    _sampled_fields = fields.sample(points[:, i_ppsk], LoS_directions[:, i_ppsk])
    _sampled_vLoS = _sampled_fields['HI']["LoS_velocity"][0]
    lbl = rf"$v_\mathrm{{LOS}}/c = {_sampled_vLoS:.2g}$"
    temp_handles.append(Line2D([0], [0], color=colors[i_ppsk], lw=4, linestyle="-", label=lbl))
leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
ax.add_artist(leg_T)



# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()


# Zoom around each line center [Angstrom]
delta_lambda_A = 1.3   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    for i_ppsk in range(N_ppsk):
        profile = phi_np[i_sp, i_sk, i_ppsk, mask]  # (N_lambda_window,)
        ax.plot(lam_win, profile, color=colors[i_ppsk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_yscale("log")
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")
axes[0].set_ylabel(r"$\phi(\lambda)\; \left[\mathrm{ns}\right]$")

fig.tight_layout()
plt.show()